# 02. Feature Engineering & Leakage Verification

Engineering temporal, cyclical, lag, rolling, and physical domain features with zero data leakage.


In [ ]:
import os
import sys
sys.path.insert(0, '..')
import pandas as pd
from src.data.loader import DataLoader
from src.data.merger import DataMerger
from src.features.feature_engineering import FeatureEngineer, FeatureLeakageChecker


## 1. Load Merged Data


In [ ]:
loader = DataLoader()
w = loader.load_weather('../data/raw/weather/nasa_power_weather.csv')
e = loader.load_energy('../data/raw/energy/solar_energy_generation.csv')
merged_df, _ = DataMerger().merge(w, e)


## 2. Generate Temporal, Physical Domain, and Lag Features (Horizon = 24h)


In [ ]:
feat_eng = FeatureEngineer(target_column='energy_output_kwh', horizon_hours=24)
df_eng, feat_cols = feat_eng.engineer_features(merged_df, output_feature_list_path='../reports/metrics/feature_list.csv')
print(f'Generated {len(feat_cols)} features:', feat_cols[:10])


## 3. Verify Zero Data Leakage


In [ ]:
FeatureLeakageChecker.verify_no_target_in_features(feat_cols, 'energy_output_kwh', 24)
FeatureLeakageChecker.verify_rolling_stats_historical(df_eng, 'energy_output_kwh', 6)
FeatureLeakageChecker.verify_target_shift_alignment(df_eng, 'energy_output_kwh', 24)
print('All feature leakage checks passed successfully!')
